# 19b — SIH Mandatory Scenario Tests

**SIH PS 26168 — Intelligent Dead Reckoning**

## Mandatory Scenarios

| Scenario | Description | SIH Target |
|----------|-------------|------------|
| **A** | ~50m / 3–5s GNSS-denied | Final error < 5 m |
| **B** | ~1km / ~60s GNSS-denied | Final error < 100 m |

## Segment Selection Rules (applied BEFORE evaluating metrics)

These rules are defined once and frozen — no cherry-picking:

**Scenario A (50m / 3-5s):**  
Find all contiguous segments in S1 where:
- Duration: 3s ≤ T ≤ 5s
- Distance traveled: 40m ≤ d ≤ 60m
- Mean speed ≥ 5 m/s (i.e., vehicle actually moving)
→ Report ALL qualifying segments. If none: report NOT_TESTABLE.

**Scenario B (1km / 60s):**  
Find all contiguous segments in S1 where:
- Duration: 55s ≤ T ≤ 65s  
- Distance traveled: 900m ≤ d ≤ 1100m
→ Report ALL qualifying segments. If none: report NOT_TESTABLE.

> ⚠️ RULE: If no qualifying segment exists in available data, the result is **NOT_TESTABLE_ON_AVAILABLE_DATA** — not fabricated.
> No results will be manually selected after looking at performance.

In [ ]:
import os, sys, json, datetime
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

RESULTS_DIR = PROJECT_ROOT / 'results' / 'sih_scenarios'
PLOTS_DIR   = PROJECT_ROOT / 'plots'   / 'sih_scenarios'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load Driver A (S1) ground truth trajectory ───────────────────────────────
from src.preprocessing.data_loader import IOVNBDLoader

loader = IOVNBDLoader()
sess = loader.load_session('S1', preprocess_imu=False)  # GT positions only

gnss_lat  = sess['gps']['lat']
gnss_lon  = sess['gps']['lon']
gnss_spd  = sess['gps']['speed_mps'] if sess['gps']['speed_mps'] is not None else np.zeros(len(gnss_lat))

# ENU ground truth
enu_gt = sess['enu_coords'][:, :2]  # (N, 2)
N = len(enu_gt)
DT = 0.1  # 10 Hz

print(f'Session S1: {N} samples, {N*DT:.1f}s, {N*DT/60:.1f}min')
print(f'Speed range: [{gnss_spd.min():.1f}, {gnss_spd.max():.1f}] m/s')

# Cumulative distance along GT trajectory
d_steps = np.linalg.norm(np.diff(enu_gt, axis=0), axis=1)
cum_dist = np.concatenate([[0.0], np.cumsum(d_steps)])
total_dist = cum_dist[-1]
print(f'Total GT distance: {total_dist:.1f} m  ({total_dist/1000:.2f} km)')

In [ ]:
# ── Segment finder utility ────────────────────────────────────────────────────

def find_qualifying_segments(enu_gt, cum_dist, speed_arr, dt,
                              t_min_s, t_max_s, d_min_m, d_max_m,
                              min_mean_speed_mps=0.0, max_segments=20):
    """
    Find all contiguous (non-overlapping) windows in the trajectory
    satisfying the given duration and distance constraints.

    Selection is purely temporal / distance based — NO evaluation of
    model performance is done here. This function is called once and
    its output is frozen before running the pipeline.
    """
    n_min = int(t_min_s / dt)
    n_max = int(t_max_s / dt)
    qualifying = []

    i = 0
    while i < len(enu_gt) - n_max:
        found = False
        for win in range(n_min, n_max + 1):
            j = i + win
            if j >= len(enu_gt):
                break
            seg_dist = cum_dist[j] - cum_dist[i]
            seg_t    = win * dt
            if d_min_m <= seg_dist <= d_max_m:
                mean_spd = float(np.mean(speed_arr[i:j]))
                if mean_spd >= min_mean_speed_mps:
                    qualifying.append({
                        'start_idx':     i,
                        'end_idx':       j,
                        'duration_s':    round(seg_t, 1),
                        'distance_m':    round(float(seg_dist), 2),
                        'mean_speed_mps': round(mean_spd, 2),
                        'start_t_s':     round(i * dt, 1),
                        'end_t_s':       round(j * dt, 1),
                    })
                    i = j  # skip to end of this window (non-overlapping)
                    found = True
                    break
        if not found:
            i += 1
        if len(qualifying) >= max_segments:
            break

    return qualifying


# ── Find Scenario A segments (50m / 3-5s) ───────────────────────────────────
print('=== Scenario A: ~50m / 3-5s ===')
segs_A = find_qualifying_segments(
    enu_gt, cum_dist, gnss_spd, dt=DT,
    t_min_s=3.0, t_max_s=5.0,
    d_min_m=40.0, d_max_m=60.0,
    min_mean_speed_mps=5.0
)
print(f'Found {len(segs_A)} qualifying segments:')
for i, s in enumerate(segs_A[:5]):
    print(f'  [{i}] t={s["start_t_s"]}–{s["end_t_s"]}s  dist={s["distance_m"]}m  spd={s["mean_speed_mps"]}m/s')

# ── Find Scenario B segments (1km / ~60s) ────────────────────────────────────
print('\n=== Scenario B: ~1km / ~60s ===')
segs_B = find_qualifying_segments(
    enu_gt, cum_dist, gnss_spd, dt=DT,
    t_min_s=55.0, t_max_s=65.0,
    d_min_m=900.0, d_max_m=1100.0,
    min_mean_speed_mps=0.0
)
print(f'Found {len(segs_B)} qualifying segments:')
for i, s in enumerate(segs_B[:5]):
    print(f'  [{i}] t={s["start_t_s"]}–{s["end_t_s"]}s  dist={s["distance_m"]}m  spd={s["mean_speed_mps"]}m/s')

In [ ]:
# ── Pipeline runner for a GNSS-denied segment ────────────────────────────────
from src.integration.final_navigation_pipeline import FinalNavigationPipeline
from src.preprocessing.data_loader import IOVNBDLoader
from src.calibration.alignment import PhoneVehicleAlignment

def run_gnss_denied_segment(sess, seg_info, use_fixed_nio=True, label=''):
    """
    Simulate GNSS blackout over a segment. Compares GT vs pipeline output.
    Returns a metrics dict with final error, RMSE, drift %, recovery jump.
    """
    i0, i1 = seg_info['start_idx'], seg_info['end_idx']

    # Use fixed NIO checkpoint if available, else baseline
    FIXED_CKPT = PROJECT_ROOT / 'checkpoints' / 'nio_fixed' / 'nio_fixed_best.pt'
    fixed_knet = PROJECT_ROOT / 'checkpoints' / 'kalmannet_fixed_input' / 'kalmannet_best.pt'
    base_knet  = PROJECT_ROOT / 'checkpoints' / 'kalmannet' / 'kalmannet_best.pt'
    KNET_CKPT  = fixed_knet if fixed_knet.exists() else base_knet

    nio_ckpt  = str(FIXED_CKPT) if (use_fixed_nio and FIXED_CKPT.exists()) else None
    knet_ckpt = str(KNET_CKPT)  if KNET_CKPT.exists() else None

    pipeline = FinalNavigationPipeline(
        knet_checkpoint_path=knet_ckpt,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

    # Align phone to vehicle frame
    aligner = PhoneVehicleAlignment()
    veh_spd = sess['vehicle']['speed_mps'] if sess['vehicle']['speed_mps'] is not None else sess['gps']['speed_mps']
    R_p2v = aligner.calibrate(sess['accel_raw'], zupt_mask=sess['zupt_mask'], velocity_ref=veh_spd)

    # Initialize at start of segment with known GT position
    lat0 = sess['gps']['lat'][i0]
    lon0 = sess['gps']['lon'][i0]
    pipeline.initialize(lat0=lat0, lon0=lon0, R_p2v=R_p2v,
                        initial_speed=float(sess['gps']['speed_mps'][i0]) if veh_spd is not None else 0.0)

    enu_gt_seg = sess['enu_coords'][i0:i1, :2]

    estimated_positions = []

    for k in range(i1 - i0):
        idx = i0 + k
        state = pipeline.step(
            accel_raw=sess['accel_filtered'][idx],
            gyro_raw=sess['gyro_filtered'][idx],
            p_gnss_geodetic=None,  # BLACKOUT for entire segment
            hdop=1.0,
            is_blackout=True,
            speed_ref=None,
            dt=0.1
        )
        estimated_positions.append([state['east_m'], state['north_m']])

    est = np.array(estimated_positions)
    gt  = enu_gt_seg - enu_gt_seg[0]   # relative to segment start
    est_rel = est - est[0]

    pos_errors = np.linalg.norm(est_rel - gt, axis=1)
    final_error = float(pos_errors[-1])
    rmse = float(np.sqrt(np.mean(pos_errors**2)))
    max_error = float(pos_errors.max())

    # Distance traveled (GT)
    dist_gt = float(np.sum(np.linalg.norm(np.diff(gt, axis=0), axis=1)))
    drift_pct = (final_error / max(dist_gt, 0.1)) * 100.0

    # Recovery jump: distance jump when GNSS returns (simulated)
    # Here: compare estimated position at end with GT position at end
    recovery_jump = final_error  # same as final error for cold reconnect

    metrics = {
        'label':             label,
        'scenario':          seg_info,
        'nio_checkpoint':    nio_ckpt if nio_ckpt else 'NOT_USED',
        'knet_checkpoint':   knet_ckpt if knet_ckpt else 'NOT_USED',
        'final_error_m':     round(final_error, 3),
        'rmse_m':            round(rmse, 3),
        'max_error_m':       round(max_error, 3),
        'distance_gt_m':     round(dist_gt, 2),
        'drift_pct':         round(drift_pct, 3),
        'recovery_jump_m':   round(recovery_jump, 3),
        'sih_target_m':      None,  # set per scenario
        'sih_pass':          None,
        'timestamp':         datetime.datetime.utcnow().isoformat() + 'Z'
    }

    return metrics, (gt, est_rel, pos_errors)

print('Pipeline runner ready.')
print('Re-loading S1 with IMU preprocessing for simulation...')
sess_imu = loader.load_session('S1', preprocess_imu=True)

In [ ]:
# ── Run Scenario A ───────────────────────────────────────────────────────────
scenario_a_results = []

if not segs_A:
    print('NOT_TESTABLE_ON_AVAILABLE_DATA: No qualifying 50m/3-5s segments found in S1')
    scenario_a_results = [{'result': 'NOT_TESTABLE_ON_AVAILABLE_DATA',
                           'reason': 'No contiguous segment with 40-60m distance and 3-5s duration at ≥5m/s found in S1'}]
else:
    print(f'Running Scenario A on {len(segs_A)} qualifying segments...')
    for i, seg in enumerate(segs_A[:3]):  # evaluate up to 3 segments
        try:
            m, pdata = run_gnss_denied_segment(sess_imu, seg,
                                               label=f'ScenarioA_seg{i}')
            m['sih_target_m'] = 5.0
            m['sih_pass']     = m['final_error_m'] < 5.0
            scenario_a_results.append(m)
            status = 'PASS ✓' if m['sih_pass'] else 'FAIL ✗'
            print(f'  Seg {i}: final_error={m["final_error_m"]}m  dist={m["distance_gt_m"]}m  drift={m["drift_pct"]:.1f}%  [{status}]')
        except Exception as e:
            print(f'  Seg {i}: ERROR — {e}')
            scenario_a_results.append({'label': f'ScenarioA_seg{i}', 'error': str(e)})

print('\n=== Scenario A Summary ===')
for r in scenario_a_results:
    if 'final_error_m' in r:
        print(f'  {r["label"]}: {r["final_error_m"]}m  SIH_PASS={r["sih_pass"]}')
    else:
        print(f'  {r}')

In [ ]:
# ── Run Scenario B ───────────────────────────────────────────────────────────
scenario_b_results = []

if not segs_B:
    print('NOT_TESTABLE_ON_AVAILABLE_DATA: No qualifying 1km/60s segments found in S1')
    scenario_b_results = [{'result': 'NOT_TESTABLE_ON_AVAILABLE_DATA',
                           'reason': 'No contiguous segment with 900-1100m distance at 55-65s duration found in S1'}]
else:
    print(f'Running Scenario B on {len(segs_B)} qualifying segments...')
    for i, seg in enumerate(segs_B[:3]):
        try:
            m, pdata = run_gnss_denied_segment(sess_imu, seg,
                                               label=f'ScenarioB_seg{i}')
            m['sih_target_m'] = 100.0
            m['sih_pass']     = m['final_error_m'] < 100.0
            scenario_b_results.append(m)
            status = 'PASS ✓' if m['sih_pass'] else 'FAIL ✗'
            print(f'  Seg {i}: final_error={m["final_error_m"]}m  dist={m["distance_gt_m"]}m  drift={m["drift_pct"]:.1f}%  [{status}]')
        except Exception as e:
            print(f'  Seg {i}: ERROR — {e}')
            scenario_b_results.append({'label': f'ScenarioB_seg{i}', 'error': str(e)})

print('\n=== Scenario B Summary ===')
for r in scenario_b_results:
    if 'final_error_m' in r:
        print(f'  {r["label"]}: {r["final_error_m"]}m  SIH_PASS={r["sih_pass"]}')
    else:
        print(f'  {r}')

In [ ]:
# ── Save all scenario results ─────────────────────────────────────────────────
output = {
    'experiment': 'SIH_mandatory_scenarios',
    'timestamp':  datetime.datetime.utcnow().isoformat() + 'Z',
    'session':    'S1',
    'driver':     'Driver A (held-out test)',
    'segment_selection_methodology': {
        'scenario_A': 'Non-overlapping windows: 3-5s, 40-60m, mean_speed≥5m/s. Applied before evaluation.',
        'scenario_B': 'Non-overlapping windows: 55-65s, 900-1100m. Applied before evaluation.'
    },
    'scenario_A_50m_3to5s': {
        'sih_target_final_error_m': 5.0,
        'segments_found':           len(segs_A),
        'results':                  scenario_a_results
    },
    'scenario_B_1km_60s': {
        'sih_target_final_error_m': 100.0,
        'segments_found':           len(segs_B),
        'results':                  scenario_b_results
    }
}

out_path = RESULTS_DIR / 'sih_scenario_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved: {out_path}')

# ── Final compliance print ────────────────────────────────────────────────────
print('\n=== SIH MANDATORY SCENARIO COMPLIANCE ===')
print(f'  Scenario A (~50m/3-5s, target <5m):')
if not segs_A:
    print('    NOT_TESTABLE_ON_AVAILABLE_DATA')
else:
    for r in scenario_a_results:
        if 'final_error_m' in r:
            s = 'PASS' if r['sih_pass'] else 'FAIL'
            print(f'    {r["label"]}: {r["final_error_m"]}m → {s}')

print(f'\n  Scenario B (~1km/60s, target <100m):')
if not segs_B:
    print('    NOT_TESTABLE_ON_AVAILABLE_DATA')
else:
    for r in scenario_b_results:
        if 'final_error_m' in r:
            s = 'PASS' if r['sih_pass'] else 'FAIL'
            print(f'    {r["label"]}: {r["final_error_m"]}m → {s}')